# Do LFP motifs track neuron groups in spontaneous awake activity? Original vs relaxed vs every tracker output

Validation of Mishler et al. (2026, Fig. 3) on every recording with Neuropixels units, then the question that
matters for us: **is the tracker more informative about spiking populations than the original algorithm?**

**Data.** 6 recordings, 5 mice, 47 region signals (each region with ≥ 20 well-isolated units and an LFP channel
in that region). Only awake, stimulation-free seconds: every electrical and visual block is cut with a 10-s
margin; the psilocybin session is cut before the psilocybin injection, with the saline injection removed.
Code: `src/awake_grounding.py`; one pickle per signal in `data/processed/awake_grounding/`.

**Spike data.** The Allen NWB files hold spike-sorted Neuropixels units (spike times, region of the peak channel,
quality metrics). Units pass the Allen quality filters (ISI violations < 0.5, amplitude cutoff < 0.1, presence
ratio > 0.9); spikes are counted in 1-s bins on the LFP seconds and smoothed with a 1-s Gaussian. The LFP is the
middle valid channel of the same region on the same probe.

**The methods compared.** Each gives one or more per-second signals per motif:
- **original**: getMotifs.m rules on the joined awake data; expression = sign-binarized z-scored envelopes
  projected on the template (Mishler's "LFP projection").
- **relaxed**: 5-min chunks, permutation p ≤ 0.05, present in half the chunks; same projection.
- **tracker**: envelopes z-scored with the mean / SD of awake spontaneous activity (the tracker's default now),
  then split by a 5-min sliding baseline. Motifs are found in sliding 5-min windows of the fast part and linked
  into a library. Its outputs, each tested separately:
  - *expression*: the fast part projected on the motif;
  - *baseline*: the slow part (5-min baseline) projected on the motif;
  - *spread*: how strongly the expression swings (its rolling SD);
  - *presence*: share of the 5-min windows around each second in which the motif was detected;
  - *unsplit*: the z-scored envelopes projected on the tracker's motif with no baseline split (a control: tracker
    direction, original-style signal).

### Mishler's claims
- **C1, the motif's poles have their own neurons.** Each neuron is assigned to the motif pole (motif k, + or −)
  whose signal its firing correlates with most. The + ensemble should fire more when the + pole is expressed;
  the − ensemble should fire less.
- **C2, the two ensembles oppose each other.** + and − ensembles of the same motif should be anticorrelated.
- **C3, a motif beats any single band.** A motif should track its neurons better than one frequency band does.
  Here each band gets the same chance: it defines its own + / − ensembles, scored the same way; for each tracker
  output the band version of the same quantity is used (band residual, band baseline, band spread).
- **C4, the split is visible in spiking alone.** A split of the neurons from their own activity (PCA) should agree
  with the LFP-defined split (paper: 72 %).

### How a relationship is judged real
1. *Held out*: neurons are assigned on 4/5 of the data (interleaved 60-s blocks) and scored on the other 1/5,
   with 10 s trimmed at block edges.
2. *Shift null*: the whole procedure, assignment included, is repeated 200 times after circularly shifting the
   LFP signal against the spikes by at least 5 min (a third of the data for the two short recordings; the same
   for every method). Shifting keeps each signal's own slowness, so slow outputs are not favoured. p < 0.05 counts.
3. Repeated on rates with the **population mean regressed out**, so "everyone fires more" cannot drive a result.

### B. Which method is most informative about spiking?
Each method's per-second outputs (all motifs together) predict, by held-out ridge regression, every neuron's rate
and the population rate. Scores: held-out R², and the share of neurons beating their own shift null. Controls:
the 10 bands, and the 10 bands split into fast + slow parts the tracker's way but with no motifs.

In [1]:
import sys, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import binomtest, wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import awake_grounding as ag
from src.state_motifs import BANDS

OUT = REPO / "data" / "out" / "2026-11-15_awake_grounding"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
for f in FIG.glob("*.png"): f.unlink()
for f in OUT.glob("*.csv"): f.unlink()
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
METHODS = ["original", "relaxed"] + [f"tracker {o}" for o in ag.TRACKER_OUTPUTS]
COL = dict(zip(METHODS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]))
SHORT = {m: m.replace("tracker ", "tracker\n") for m in METHODS}
NULL = "#9a9a96"
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
BAND_LAB = [f"{lo:g}–{hi:g}" for lo, hi in BANDS]
rng = np.random.default_rng(0)
def frac(p): return f"{(p < .05).sum()}/{p.notna().sum()}"

R = [ag.load(f) for f in sorted(glob.glob(str(REPO / "data/processed/awake_grounding/*.pkl")))]
META = pd.DataFrame([r["meta"] for r in R])
C = pd.concat([r["claims"] for r in R if len(r["claims"])], ignore_index=True)
C["signal"] = C.session + " " + C.region
INFO = pd.concat([r["info"] for r in R], ignore_index=True)
META.to_csv(OUT / "signals.csv", index=False)
C.drop(columns="band_r_diff").to_csv(OUT / "claims_per_motif.csv", index=False)
INFO.to_csv(OUT / "B_information_per_signal.csv", index=False)
print(f"{len(R)} signals, {META.session.nunique()} recordings, {META.n_units.sum()} units")
META.groupby("session").agg(signals=("region", "size"), units=("n_units", "sum"), awake_seconds=("n_seconds", "first"),
                            how=("how", "first"))

65 signals, 8 recordings, 2629 units


,signals,units,awake_seconds,how
session,,,,
688277-20230921,6,239,350,before isoflurane on (-30 s)
692644-20230906,10,343,337,before isoflurane on (-30 s)
692644-20230907,7,310,1115,before psilocybin injection; saline injection ...
703063-20231116,7,373,845,before isoflurane on (-30 s)
703064-20231129,8,264,863,before isoflurane on (-30 s)
703065-20231207,10,346,1049,before isoflurane on (-30 s)
709401-20231213,9,448,866,before isoflurane on (-30 s)
720762-20240221,8,306,928,before isoflurane on (-30 s)


## 0. Coverage: how much awake data, and which signals have motifs
The tracker needs ≥ ~7 min (four overlapping 5-min windows): it finds nothing in the two ~6-min recordings.

In [2]:
cov = META.melt(id_vars=["session", "region"], value_vars=["n_motifs_original", "n_motifs_relaxed", "n_motifs_tracker"],
                var_name="pipeline", value_name="motifs")
cov["pipeline"] = cov.pipeline.str.replace("n_motifs_", "")
tbl = cov.groupby(["session", "pipeline"]).motifs.apply(lambda m: f"{(m > 0).sum()}/{m.size}").unstack()[["original", "relaxed", "tracker"]]
tbl.insert(0, "awake min", (META.groupby("session").n_seconds.first() / 60).round(1))
tbl.to_csv(OUT / "0_coverage.csv"); tbl

pipeline,awake min,original,relaxed,tracker
session,,,,
688277-20230921,5.8,5/6,5/6,0/6
692644-20230906,5.6,9/10,9/10,0/10
692644-20230907,18.6,7/7,7/7,7/7
703063-20231116,14.1,7/7,7/7,7/7
703064-20231129,14.4,8/8,8/8,8/8
703065-20231207,17.5,10/10,10/10,10/10
709401-20231213,14.4,7/9,9/9,7/9
720762-20240221,15.5,8/8,2/8,3/8


## 1. Example of C1 in one signal
692644 (pre-psilocybin awake), AV thalamus, original motif. Black: motif expression. Coloured: held-out mean rate
of the neurons assigned to its + pole and to its − pole (each chosen on the other 4/5 of the data).

In [3]:
ex = next(r for r in R if r["meta"]["session"] == "692644-20230907" and r["meta"]["region"] == "AV")
fid, inner = ag.folds(ex["rate"].shape[0])
x = ex["outputs"]["original"][:, 0]
A, B, _ = ag.heldout_ensembles(x[:, None], ex["rate"], fid, inner)
z = lambda v: (v - np.nanmean(v)) / np.nanstd(v)
fig, ax = plt.subplots(figsize=(11, 3))
tt = np.arange(x.size)
ax.plot(tt, z(np.where(inner, x, np.nan)), color="0.15", lw=1.2, label="motif expression (LFP)")
ax.plot(tt, z(A[:, 0]), color="#2a78d6", lw=1, label="+ ensemble rate (held out)")
ax.plot(tt, z(B[:, 0]), color="#eb6834", lw=1, label="− ensemble rate (held out)")
row = C[(C.signal == "692644-20230907 AV") & (C.method == "original") & (C.rates == "raw")].iloc[0]
ax.set_title(f"r(expression, + ensemble) = {row.r_plus:.2f} (p = {row.p_plus:.3f}); "
             f"r(expression, − ensemble) = {row.r_minus:.2f} (p = {row.p_minus:.3f})", fontsize=9, loc="left")
ax.set_xlabel("awake, stimulation-free seconds (concatenated)"); ax.set_ylabel("z")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.25))
save(fig, "1_example_C1"); plt.show()

## 2. C1 for every method and every tracker output
Score: held-out r between the motif signal and (+ ensemble − − ensemble). One dot per motif per signal; grey
tick = 95th percentile of its shift null; filled = p < 0.05.

In [4]:
summ = []
for rates in ("raw", "population removed"):
    for m in METHODS:
        d = C[(C.rates == rates) & (C.method == m)].dropna(subset=["r_diff"])
        k = int((d.p_diff < .05).sum())
        summ.append({"rates": rates, "method": m, "motifs": len(d), "signals": d.signal.nunique(),
                     "C1 + ensemble up": frac(d.p_plus), "C1 − ensemble down": frac(d.p_minus),
                     "C1 (+ minus −)": frac(d.p_diff), "share": round(k / max(len(d), 1), 2),
                     "binomial p vs 5 %": binomtest(k, len(d), .05, alternative="greater").pvalue if len(d) else np.nan,
                     "median r": d.r_diff.median(), "non-broadband passing": frac(d[~d.broadband.astype(bool)].p_diff)})
S1 = pd.DataFrame(summ); S1.to_csv(OUT / "2_C1_summary.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True)
for ax, rates in zip(axes, ("raw", "population removed")):
    for k, m in enumerate(METHODS):
        d = C[(C.rates == rates) & (C.method == m)].dropna(subset=["r_diff"])
        xs = k + rng.uniform(-0.2, 0.2, len(d))
        ax.scatter(xs, d.null95_diff, marker="_", s=50, color=NULL, lw=1.1, zorder=2)
        ax.scatter(xs, d.r_diff, s=20, color=np.where(d.p_diff < .05, COL[m], "white"), edgecolor=COL[m], lw=1, zorder=3)
        ax.text(k, 1.03, f"{(d.p_diff < .05).sum()}/{len(d)}", ha="center", fontsize=8.5, color="0.2")
    ax.axhline(0, color="0.75", lw=0.6); ax.axvline(1.5, color="0.85", lw=0.8)
    ax.set_ylim(-0.45, 1.12); ax.set_ylabel("held-out r(signal, + ens − − ens)")
    ax.set_title(f"rates: {rates}   (numbers = motifs passing / tested)", fontsize=9, loc="left")
axes[1].set_xticks(range(len(METHODS)), [SHORT[m] for m in METHODS])
fig.suptitle("C1: does each method's motif signal pick out neurons that follow it on held-out data?", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "2_C1_per_method"); plt.show()
S1.round(3)

,rates,method,motifs,signals,C1 + ensemble up,C1 − ensemble down,C1 (+ minus −),share,binomial p vs 5 %,median r,non-broadband passing
0,raw,original,70,61,54/70,53/70,60/70,0.86,0.0,0.569,50/58
1,raw,relaxed,70,56,53/70,48/70,56/70,0.80,0.0,0.544,46/55
2,raw,tracker expression,77,42,58/77,50/77,61/77,0.79,0.0,0.486,43/56
3,raw,tracker baseline,80,42,35/80,42/80,40/80,0.50,0.0,0.369,32/60
4,raw,tracker spread,80,42,15/80,24/80,25/80,0.31,0.0,0.181,15/59
5,raw,tracker presence,62,31,18/62,20/62,14/62,0.23,0.0,0.285,8/44
6,raw,tracker unsplit,75,42,57/75,55/75,63/75,0.84,0.0,0.510,44/55
7,population removed,original,71,61,55/71,54/71,57/71,0.80,0.0,0.456,49/59
8,population removed,relaxed,73,57,60/73,62/73,63/73,0.86,0.0,0.430,47/57
9,population removed,tracker expression,83,42,68/83,69/83,69/83,0.83,0.0,0.410,51/62


## 3. C2: are + and − ensembles more anticorrelated than the assignment alone makes them?
Splitting neurons by the sign of their correlation with any signal makes two groups that anticorrelate, even for
an unrelated (shifted) signal. Only beating that counts. Raw rates only (with the population mean removed, any
split is pushed towards −1).

In [5]:
d = C[C.rates == "raw"].dropna(subset=["r_ens"])
S2 = d.groupby("method").agg(motifs=("r_ens", "size"), median_r=("r_ens", "median"),
                             median_null=("null_mean_ens", "median"), passing=("p_ens", frac)).reindex(METHODS)
S2.to_csv(OUT / "3_C2_summary.csv")
fig, ax = plt.subplots(figsize=(5.4, 4.8))
for m in ["original", "relaxed", "tracker expression", "tracker baseline"]:
    e = d[d.method == m]
    ax.scatter(e.null_mean_ens, e.r_ens, s=22, color=np.where(e.p_ens < .05, COL[m], "white"), edgecolor=COL[m], lw=1,
               label=f"{m}: {(e.p_ens < .05).sum()}/{len(e)}")
ax.plot([-1, 1], [-1, 1], color="0.6", lw=0.8, ls="--")
ax.set_xlim(-0.85, 0.65); ax.set_ylim(-0.95, 0.85)
ax.set_xlabel("r(+ ens, − ens) expected from the assignment alone\n(mean of shift null)")
ax.set_ylabel("r(+ ens, − ens) observed, held out")
ax.legend(frameon=False, fontsize=7.5, loc="upper left", title="filled = beats null", title_fontsize=7.5)
ax.set_title("C2: below the dashed line = more anticorrelated than chance", fontsize=9, loc="left")
save(fig, "3_C2_opponent_ensembles"); plt.show()
S2.round(2)

,motifs,median_r,median_null,passing
method,,,,
original,70,-0.26,-0.10,18/70
relaxed,70,-0.23,-0.09,21/70
tracker expression,77,-0.06,-0.09,11/77
tracker baseline,80,-0.21,-0.08,8/80
tracker spread,80,-0.09,-0.06,5/80
tracker presence,62,-0.09,-0.06,3/62
tracker unsplit,75,-0.18,-0.10,15/75


## 4. C3: does a motif track neurons better than a single band?
Share of the 10 bands the motif beats (0.5 = as good as a typical band; Mishler: ~0.9). Each tracker output is
compared with the same quantity for single bands; presence has no band version.

In [6]:
d = C[C.rates == "raw"].dropna(subset=["frac_bands_beaten"])
ms = [m for m in METHODS if m != "tracker presence"]
fig, ax = plt.subplots(figsize=(9, 3.6))
for k, m in enumerate(ms):
    e = d[d.method == m]
    ax.scatter(k + rng.uniform(-0.2, 0.2, len(e)), e.frac_bands_beaten, s=18, color=COL[m], alpha=0.85)
    ax.plot([k - 0.28, k + 0.28], [e.frac_bands_beaten.mean()] * 2, color="0.15", lw=2)
    ax.text(k, 1.07, f"{e.frac_bands_beaten.mean():.2f}", ha="center", fontsize=8.5)
ax.axhline(0.5, color="0.6", lw=0.8, ls="--")
ax.set_xticks(range(len(ms)), [SHORT[m] for m in ms]); ax.set_ylim(-0.05, 1.14)
ax.set_ylabel("share of bands beaten"); ax.set_title("C3 (bar and number = mean)", fontsize=9, loc="left")
save(fig, "4_C3_motif_vs_bands"); plt.show()
S3 = d.groupby("method").agg(motifs=("r_diff", "size"), mean_share_beaten=("frac_bands_beaten", "mean"),
                             beats_every_band=("beats_best_band", "mean")).reindex(ms)
S3.to_csv(OUT / "4_C3_summary.csv")
print("best single band per signal:", C.drop_duplicates("signal").best_band.map(lambda i: BAND_LAB[i] if i >= 0 else "-").value_counts().to_dict())
S3.round(2)

best single band per signal: {'50–80': 15, '1–4': 11, '8–12': 11, '3–6': 7, '35–55': 6, '10–15': 5, '25–40': 3, '5–9': 3, '13–20': 2, '18–30': 1}


,motifs,mean_share_beaten,beats_every_band
method,,,
original,70,0.71,0.228571
relaxed,70,0.63,0.185714
tracker expression,77,0.62,0.207792
tracker baseline,80,0.48,0.125
tracker spread,80,0.30,0.025
tracker unsplit,75,0.63,0.253333


## 5. C4: is the split visible in spiking alone?
**PCA (Mishler's test).** For each motif, the neurons the LFP assigned to it; PCA on their z-scored rates; split
by the sign of their first-component loading; agreement with their + / − assignment. Null: labels permuted
(chance is above 0.5 because the better of the two labelings is taken).

**getClust** (Mishler's ensemble code, `src/get_clust.py`), from spiking alone:
1. each neuron's rate is squared and z-scored;
2. for a candidate group, *support* each second = (share of members active) × (their mean activity); stretches
   above mean + 1 SD lasting ≥ 3 s are the group's *events*;
3. *recruitment* = a neuron's activity during events minus outside them; neurons below a rising threshold
   (0.3, 0.6, 1.0) are dropped and events recomputed until membership is stable; leftovers seed the next group;
4. each group (≥ 5 neurons) is checked 500 times: events from 3 random members, the others must be reliably
   recruited (mean / SD > 1);
5. groups that recruit each other (> 0.6) are merged.

Then: does each getClust group's activity follow a motif signal? Same shift null.

In [7]:
P = pd.concat([r["pca"] for r in R], ignore_index=True).dropna(subset=["agreement"])
G = pd.concat([r["getclust"] for r in R if len(r["getclust"])], ignore_index=True).dropna(subset=["r"])
gbest = G.loc[G.groupby(["session", "region", "cluster", "method"]).r.apply(lambda r: r.abs().idxmax())]
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax = axes[0]
for k, m in enumerate(METHODS):
    e = P[P.method == m]
    xs = k + rng.uniform(-0.2, 0.2, len(e))
    ax.scatter(xs, e.null_mean, marker="_", s=50, color=NULL, lw=1.1)
    ax.scatter(xs, e.agreement, s=20, color=np.where(e.p < .05, COL[m], "white"), edgecolor=COL[m], lw=1)
    ax.text(k, 1.04, f"{e.agreement.median():.2f}  ({(e.p < .05).sum()}/{len(e)})", ha="center", fontsize=8)
ax.axhline(0.72, color="0.4", lw=0.8, ls=":"); ax.text(6.45, 0.72, "Mishler 72 %", fontsize=7.5, va="center")
ax.set_ylim(0.4, 1.12); ax.set_ylabel("agreement, PCA split\nvs LFP split")
ax.set_title("PCA on spiking alone: median agreement (passing / tested); grey tick = permutation null mean", fontsize=9, loc="left")
ax = axes[1]
for k, m in enumerate(METHODS):
    e = gbest[gbest.method == m]
    ax.scatter(k + rng.uniform(-0.2, 0.2, len(e)), e.r.abs(), s=np.clip(e["size"], 5, 60),
               color=np.where(e.p < .05, COL[m], "white"), edgecolor=COL[m], lw=1, alpha=0.9)
    ax.text(k, 0.86, f"{(e.p < .05).sum()}/{len(e)}", ha="center", fontsize=8.5)
ax.set_ylim(0, 0.95); ax.set_ylabel("|r|, getClust group\nvs its best motif")
ax.set_title("getClust groups following a motif signal (dot size = group size; filled = p < 0.05)", fontsize=9, loc="left")
axes[1].set_xticks(range(len(METHODS)), [SHORT[m] for m in METHODS])
fig.tight_layout(); save(fig, "5_C4_from_spiking_alone"); plt.show()
g1 = G.drop_duplicates(["session", "region", "cluster"])
S4 = pd.DataFrame({"PCA agreement (median)": P.groupby("method").agreement.median(),
                   "PCA null (median)": P.groupby("method").null_mean.median(),
                   "PCA passing": P.groupby("method").p.apply(frac),
                   "getClust groups following": gbest.groupby("method").p.apply(frac)}).reindex(METHODS)
S4.to_csv(OUT / "5_C4_summary.csv")
print(f"getClust: {len(g1)} groups, median size {g1['size'].median():.0f}, median stability across seeds {g1.stability.median():.2f}")
S4.round(2)

getClust: 158 groups, median size 8, median stability across seeds 1.00


,PCA agreement (median),PCA null (median),PCA passing,getClust groups following
method,,,,
original,0.88,0.59,45/65,102/141
relaxed,0.88,0.60,44/60,100/140
tracker expression,0.86,0.62,32/60,62/102
tracker baseline,0.84,0.61,32/61,52/102
tracker spread,0.79,0.61,28/62,39/102
tracker presence,0.88,0.61,22/34,34/83
tracker unsplit,0.93,0.61,37/54,74/102


## 6. B: which method's outputs are most informative about spiking?
Paired over the signals where every method found at least one motif. Left: held-out R² of every neuron's rate
(mean over neurons). Middle: held-out R² of the population rate. Right: share of neurons whose R² beats their own
shift null. Grey lines join one signal.

In [8]:
FS = ["original", "relaxed", "tracker expression", "tracker all outputs", "10 bands", "10 bands, split (fast + slow)"]
FCOL = {"original": COL["original"], "relaxed": COL["relaxed"], "tracker expression": COL["tracker expression"],
        "tracker all outputs": "#0d6b4a", "10 bands": "#7a7a75", "10 bands, split (fast + slow)": "#3a3a37"}
FLAB = {"original": "original", "relaxed": "relaxed", "tracker expression": "tracker\nexpression",
        "tracker all outputs": "tracker\nall outputs", "10 bands": "10 bands\n(control)",
        "10 bands, split (fast + slow)": "10 bands split\nfast + slow\n(control)"}
pv = {v: INFO.pivot_table(index=["session", "region"], columns="features", values=v).dropna(subset=FS)[FS]
      for v in ("neuron_r2", "pop_r2", "neurons_beating_null")}
n_pair = len(pv["pop_r2"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (v, lab) in zip(axes, [("neuron_r2", "held-out R², single neurons (mean)"), ("pop_r2", "held-out R², population rate"),
                               ("neurons_beating_null", "share of neurons beating their null")]):
    M = pv[v].to_numpy()
    for row in M:
        ax.plot(range(len(FS)), row, color="0.85", lw=0.7, zorder=1)
    for k, f in enumerate(FS):
        ax.scatter(np.full(n_pair, k), M[:, k], s=16, color=FCOL[f], zorder=2)
        ax.plot([k - 0.3, k + 0.3], [np.median(M[:, k])] * 2, color="0.1", lw=2, zorder=3)
    ax.set_xticks(range(len(FS)), [FLAB[f] for f in FS], fontsize=7.5)
    ax.axvline(3.5, color="0.8", lw=0.8); ax.axhline(0, color="0.75", lw=0.6)
    ax.set_title(lab, fontsize=9, loc="left")
fig.suptitle(f"B: how much of the spiking each method's outputs explain ({n_pair} signals, paired; bar = median)",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "6_B_information"); plt.show()
tests = []
for v in pv:
    for a, b in [("tracker all outputs", "original"), ("tracker all outputs", "relaxed"), ("tracker expression", "original"),
                 ("tracker all outputs", "10 bands, split (fast + slow)")]:
        tests.append({"score": v, "comparison": f"{a} vs {b}", "median a": pv[v][a].median(), "median b": pv[v][b].median(),
                      "a wins": f"{(pv[v][a] > pv[v][b]).sum()}/{n_pair}", "Wilcoxon p": wilcoxon(pv[v][a], pv[v][b]).pvalue})
SB = pd.DataFrame(tests); SB.to_csv(OUT / "6_B_summary.csv", index=False)
SB.round(4)

,score,comparison,median a,median b,a wins,Wilcoxon p
0,neuron_r2,tracker all outputs vs original,0.2103,0.0904,35/39,0.0000
1,neuron_r2,tracker all outputs vs relaxed,0.2103,0.0930,35/39,0.0000
2,neuron_r2,tracker expression vs original,0.1004,0.0904,19/39,0.4939
3,neuron_r2,"tracker all outputs vs 10 bands, split (fast +...",0.2103,0.1955,17/39,0.9066
4,pop_r2,tracker all outputs vs original,0.3316,0.0166,32/39,0.0000
5,pop_r2,tracker all outputs vs relaxed,0.3316,0.0716,32/39,0.0000
6,pop_r2,tracker expression vs original,0.0911,0.0166,23/39,0.5206
7,pop_r2,"tracker all outputs vs 10 bands, split (fast +...",0.3316,0.3963,13/39,0.0126
8,neurons_beating_null,tracker all outputs vs original,0.7917,0.6486,25/39,0.0015
9,neurons_beating_null,tracker all outputs vs relaxed,0.7917,0.6486,23/39,0.0190


## 7. Per recording
Share of motifs passing C1 (raw rates), so no single mouse carries the result.

In [9]:
pr = C[C.rates == "raw"].dropna(subset=["r_diff"]).groupby(["session", "method"]).p_diff.apply(frac).unstack()
pr = pr.reindex(columns=METHODS); pr.to_csv(OUT / "7_C1_per_recording.csv"); pr

method,original,relaxed,tracker expression,tracker baseline,tracker spread,tracker presence,tracker unsplit
session,,,,,,,
688277-20230921,5/5,4/5,NaN,NaN,NaN,NaN,NaN
692644-20230906,7/9,9/12,NaN,NaN,NaN,NaN,NaN
692644-20230907,9/9,9/10,14/16,6/17,6/17,3/15,12/14
703063-20231116,7/7,8/10,13/13,9/12,5/12,3/7,11/11
703064-20231129,10/12,7/8,10/13,8/15,6/15,3/11,11/14
703065-20231207,11/11,10/11,16/16,8/16,3/16,0/9,16/16
709401-20231213,6/8,9/12,8/14,8/15,4/15,4/15,12/15
720762-20240221,5/9,0/2,0/5,1/5,1/5,1/5,1/5


## Conclusions
Awake, stimulation-free; 65 region signals, 8 recordings, 7 mice; tracker motifs exist in 42 signals (it cannot run
on the two ~6-min recordings).
- **C1 replicates for every method that carries the fast signal.** original 60/70 motifs, relaxed 56/70, tracker
  expression 61/77, tracker unsplit 63/75 (5 % expected); it survives removing the population mean. The slow
  tracker outputs pass less often (baseline 40/80, spread 25/80, presence 14/62): with ~15 min of data a
  5-min-scale signal has only ~3 independent stretches, so the shift null is wide. That is a power limit, not
  evidence against them.
- **C2 is mostly built into the method.** Ensembles anticorrelate, but unrelated (shifted) LFP does too; only
  3–21 motifs per method beat it.
- **C3 is weaker than reported.** Original motifs beat 71 % of bands on average (Mishler ~90 %); relaxed 63 %,
  tracker expression 62 %, unsplit 63 %, baseline 48 %, spread 30 %.
- **C4 replicates for every method and output.** PCA of spiking alone reproduces the LFP split (median
  agreement 0.79–0.93 vs ~0.60 chance; Mishler 72 %).
- **Information (B): the tracker's outputs together are more informative than the original algorithm.** On the
  39 signals where every method has motifs: population-rate R² 0.33 vs 0.02 (tracker better in 32/39), single-
  neuron R² 0.21 vs 0.09 (35/39), neurons explained above chance 79 % vs 65 % (25/39). Tracker *expression*
  alone is no better than the original; the gain comes from adding baseline, spread and presence.
- **But the motifs are not what earns the gain.** The same fast + slow split applied to the 10 raw bands, with
  no motifs, does as well for neurons and better for the population rate (R² 0.40 vs 0.33, p = 0.01). What makes
  the tracker informative about spiking is separating slow from fast activity; its motif directions compress
  (4 features per motif vs 20) but add no measurable information in awake data.